# M11 L11: Observability and Cost -- Metrics, Logs, Traces, Alerts

**Build a tiny observability + cost stack from scratch: counter, gauge, histogram, spans, structured logger, cost tracker, alerter, and dashboard.**

Duration: 90 min.  No API keys.

## Plain English (the 12-year-old version)

Every production system is **four things you can see**:

1. **Metrics** -- numbers that go up, down, or get distributed (calls, latency, queue depth)
2. **Logs** -- what happened to this one specific request (a JSON line per call)
3. **Traces** -- where the time went in this one request (a tree of "spans")
4. **Cost** -- how many dollars this request cost (token cost in, token cost out)

Get those four right and you can answer any "is the system healthy?" question in under 5 seconds. Skip any of them and you're flying blind.

## The 5 things to know

1. **The three pillars + cost** -- metrics, logs, traces, and (for LLMs) cost. Each catches a different blind spot.
2. **P50 / P95 / P99** -- mean latency hides the tail. P99 is the user who churns. Use linear interpolation.
3. **Structured logs** -- `{"level": "info", "call_id": "c0042", "model": "gpt-4o", "cost_usd": 0.0023}` is searchable. `"INFO: call ok\n"` is not.
4. **Spans form a tree** -- root span contains child spans. The trace_id is shared; the parent_id links them. Total wall-clock = max(end) - min(start).
5. **Alert severity ladder** -- info < warn < error < page. Not everything should wake up the on-call. Every alert has a runbook.

In [1]:
# === Standard setup ===
import sys, importlib.util
from pathlib import Path

def find_root(p):
    while p != p.parent:
        if (p / 'lessons').exists() and (p / 'SYLLABUS.md').exists():
            return p
        p = p.parent
    return Path.cwd()

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

_p = ROOT / 'lessons' / 'm11-observability' / 'L11-observability-cost.py'
_spec = importlib.util.spec_from_file_location('L11', _p)
_L11 = importlib.util.module_from_spec(_spec)
sys.modules['L11'] = _L11
_spec.loader.exec_module(_L11)

print(f'Setup OK. Lesson module: {_L11.LESSON_NUMBER}')

Setup OK. Lesson module: M11.L11


In [2]:
# Run the 21 unit tests inline
import inspect
tests = [(n, fn) for n, fn in inspect.getmembers(_L11, inspect.isfunction) if n.startswith('test_')]
print(f'Running {len(tests)} tests...\n')
passed, failed = 0, 0
for name, fn in tests:
    try:
        fn()
        print(f'  PASS  {name}')
        passed += 1
    except AssertionError as e:
        print(f'  FAIL  {name}: {e}')
        failed += 1
print(f'\n{passed}/{len(tests)} tests pass, {failed} fail')
assert failed == 0, f'{failed} test(s) failed -- fix before continuing'

Running 22 tests...

  PASS  test_alert_rule_fires_gt
  PASS  test_alert_rule_fires_lt
  PASS  test_alerter_evaluate_fires_matching
  PASS  test_alerter_highest_severity
  PASS  test_compute_cost_usd_known
  PASS  test_compute_cost_usd_unknown_model
  PASS  test_cost_tracker_aggregates
  PASS  test_cost_tracker_top_n
  PASS  test_counter_cannot_decrease
  PASS  test_counter_inc
  PASS  test_dashboard_summary_keys
  PASS  test_gauge_up_down
  PASS  test_histogram_empty
  PASS  test_histogram_percentiles
  PASS  test_logger_json_output
  PASS  test_logger_level_filtering
  PASS  test_metric_series_get_or_create
  PASS  test_observability_math_invariants
  PASS  test_run_pipeline_smoke
  PASS  test_simulate_call_realistic
  PASS  test_span_duration
  PASS  test_trace_root_and_children

22/22 tests pass, 0 fail


In [3]:
# === Section 1: metrics (counter, gauge, histogram) ===
ms = _L11.MetricSeries()
ms.counter('calls').inc(100)
ms.counter('errors').inc(4)
ms.gauge('queue_depth').set(12)
hist = ms.histogram('latency_ms')
for v in [120, 150, 200, 250, 300, 400, 500, 800, 1200, 2000]:
    hist.observe(v)
print(f'  counter calls      = {ms.counters["calls"].value}')
print(f'  counter errors     = {ms.counters["errors"].value}')
print(f'  gauge queue_depth  = {ms.gauges["queue_depth"].value}')
print(f'  hist  latency_ms   n={hist.count}  p50={hist.p50():.1f}  p95={hist.p95():.1f}  p99={hist.p99():.1f}')

  counter calls      = 100.0
  counter errors     = 4.0
  gauge queue_depth  = 12
  hist  latency_ms   n=10  p50=350.0  p95=1640.0  p99=1928.0


In [4]:
# === Section 2: traces (parent/child spans) ===
t = _L11.Trace('demo-t1')
root = t.start_span('pipeline', attributes={'user': 'u-alice'})
for name in ('retrieve', 'rerank', 'llm_call'):
    child = t.start_span(name, parent_id=root.span_id)
    child.finish()
root.finish()
print(f'  trace_id:   {t.trace_id}')
print(f'  root_id:    {t.root_id}')
print(f'  span_count: {t.span_count()}')
print(f'  total dur:  {t.total_duration_ms():.2f} ms')
for sid, s in list(t.spans.items())[:3]:
    print(f'    {s.name:<10}  parent={s.parent_id}  dur={s.duration_ms:.2f}ms  status={s.status}')

  trace_id:   demo-t1
  root_id:    demo-t1-s0
  span_count: 4
  total dur:  0.08 ms
    pipeline    parent=None  dur=0.08ms  status=ok
    retrieve    parent=demo-t1-s0  dur=0.00ms  status=ok
    rerank      parent=demo-t1-s0  dur=0.00ms  status=ok


In [5]:
# === Section 3: cost tracker (per-call + aggregates) ===
ct = _L11.CostTracker()
ct.add(_L11.CostRecord('c1', 0, 'gpt-4o', 'rag', 'u-alice', 1000, 500, _L11.compute_cost_usd('gpt-4o', 1000, 500), 800.0))
ct.add(_L11.CostRecord('c2', 0, 'gpt-4o-mini', 'chat', 'u-bob', 500, 200, _L11.compute_cost_usd('gpt-4o-mini', 500, 200), 200.0))
ct.add(_L11.CostRecord('c3', 0, 'claude-haiku', 'classify', 'u-alice', 800, 100, _L11.compute_cost_usd('claude-haiku', 800, 100), 150.0))
print(f'  total cost:  ${ct.total_cost_usd():.6f}')
print(f'  by model:    {[(k, round(v, 6)) for k, v in ct.by_model().items()]}')
print(f'  by user:     {[(k, round(v, 6)) for k, v in ct.by_user().items()]}')
print(f'  by feature:  {[(k, round(v, 6)) for k, v in ct.by_feature().items()]}')

  total cost:  $0.013020
  by model:    [('gpt-4o', 0.0125), ('gpt-4o-mini', 0.000195), ('claude-haiku', 0.000325)]
  by user:     [('u-alice', 0.012825), ('u-bob', 0.000195)]
  by feature:  [('rag', 0.0125), ('chat', 0.000195), ('classify', 0.000325)]


In [6]:
# === Section 4: structured logger ===
sink = []
lg = _L11.Logger(level='info', sink=sink)
lg.info('llm_call_ok', call_id='c0001', model='gpt-4o-mini', cost_usd=0.00045)
lg.info('llm_call_ok', call_id='c0002', model='gpt-4o', cost_usd=0.012)
lg.error('llm_call_failed', call_id='c0003', reason='rate_limit')
import json
for line in lg.lines():
    rec = json.loads(line)
    print(f'  {rec["level"]:>5}  {rec["message"]:<20}  {[(k,v) for k,v in rec.items() if k not in ("ts_ms","level","message")]}')

   info  llm_call_ok           [('call_id', 'c0001'), ('cost_usd', 0.00045), ('model', 'gpt-4o-mini')]
   info  llm_call_ok           [('call_id', 'c0002'), ('cost_usd', 0.012), ('model', 'gpt-4o')]
  error  llm_call_failed       [('call_id', 'c0003'), ('reason', 'rate_limit')]


In [7]:
# === Section 5: alerter (threshold rules) ===
al = _L11.Alerter(rules=[
    _L11.AlertRule('high_error_rate', 'error_rate', '>', 0.05, 'page'),
    _L11.AlertRule('slow_p95', 'p95_latency_ms', '>', 1000.0, 'error'),
    _L11.AlertRule('cost_ceiling', 'cost_usd', '>', 1.0, 'warn'),
    _L11.AlertRule('healthy_throughput', 'calls_total', '>', 0, 'info'),
])
fired = al.evaluate({
    'error_rate': 0.04,         # below threshold: no page
    'p95_latency_ms': 1200.0,   # above threshold: fires
    'cost_usd': 0.79,           # below threshold: no warn
    'calls_total': 100,         # above 0: fires
})
print(f'  rules evaluated: 4')
print(f'  alerts fired:    {len(fired)}')
for a in fired:
    print(f'    {a}')
print(f'  highest severity: {al.highest_severity()}')

  rules evaluated: 4
  alerts fired:    2
    [ERROR] slow_p95: p95_latency_ms=1200.0 > 1000.0
    [INFO ] healthy_throughput: calls_total=100 > 0
  highest severity: error


In [8]:
# === Section 6: end-to-end pipeline (100 simulated calls) ===
metrics, tracker, logger, _, trace = _L11.run_pipeline(n_calls=100, seed=_L11.SEED)
print(f'  calls:       {tracker.total_calls()}')
print(f'  errors:      {tracker.error_count()}  (rate={tracker.error_rate():.2%})')
print(f'  total cost:  ${tracker.total_cost_usd():.4f}')
lat = metrics.histograms['latency_ms']
print(f'  p50/p95/p99: {lat.p50():.1f} / {lat.p95():.1f} / {lat.p99():.1f} ms')
print(f'  trace spans: {trace.span_count()}')
print(f'  top model:   {tracker.top_n("model", 1)[0]}')

  calls:       100
  errors:      4  (rate=4.00%)
  total cost:  $0.7897
  p50/p95/p99: 360.9 / 1027.1 / 1112.7 ms
  trace spans: 101
  top model:   ('gpt-4o', 0.427685)


In [9]:
# === Section 7: dashboard summary ===
alerter = _L11.evaluate_alerts(metrics, tracker)
dash = _L11.Dashboard(metrics=metrics, tracker=tracker, alerter=alerter)
print('  Dashboard:')
dash.print_summary()
print()
print(f'  summary dict: {dash.summary()}')

  Dashboard:
    total_calls:        100
    error_count:        4  (rate=4.00%)
    p50 / p95 / p99:      360.9 /  1027.1 /  1112.8 ms
    total_cost_usd:     $0.7897
    input/output tok:   203528 / 77404
    alerts fired:       2  (max severity=error)

  summary dict: {'total_calls': 100, 'error_count': 4, 'error_rate': 0.04, 'p50_ms': 360.93, 'p95_ms': 1027.13, 'p99_ms': 1112.75, 'total_cost_usd': 0.789687, 'total_input_tokens': 203528, 'total_output_tokens': 77404, 'active_alerts': 2, 'highest_severity': 'error'}


## Architecture (Mermaid)

```mermaid
flowchart LR
    REQ[request] --> ROOT[Trace root<br/>span_id=t-19-s0]
    ROOT --> S1[Span: retriever]
    ROOT --> S2[Span: llm_call<br/>parent_id=root]
    S2 --> LOG[Logger<br/>JSON line]
    S2 --> MET[Metrics<br/>counter + histogram]
    S2 --> CST[CostTracker<br/>+ cost_usd]
    MET --> ALERT[Alerter<br/>threshold rules]
    ALERT --> DASH[Dashboard<br/>P50/P95/P99, cost]
    CST --> DASH
    LOG --> DASH
    DASH --> OP[On-call<br/>sees summary]
```

**Read as:** Every request gets a trace root; the LLM call is one span; the span emits a log, updates a histogram, and writes a cost record. The alerter evaluates threshold rules against the current values; the dashboard shows the totals; the on-call sees the dashboard.

## Reflect (10 min)

1. Why are the three pillars (metrics, logs, traces) not enough for LLM apps? What does cost add?
2. Walk through why P99 is more useful than mean latency for an SLO.
3. The cost breakdown shows gpt-4o at 54% of cost. What are two ways to cut that without changing the model?
4. An alert fires on `error_rate > 5%`. The on-call gets paged. The error rate drops to 4% an hour later. What might have been missed?
5. (Stretch) Add a `latency_buckets(hist, buckets=[100, 250, 500, 1000, 2500])` that returns the count of observations in each bucket. Why are bucketed histograms better than raw value lists for high-cardinality data?

---

**Read next:**
- `lessons/m12-security/L12-security.ipynb` -- PII, prompt injection, secrets
- `lessons/m10-evaluation/L10-evaluation-ragas.ipynb` -- RAGAS eval framework
- `SYLLABUS.md` -- the 18 modules and 21-day plan